In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# 2. Unified Data Pipeline with Augmentation & ImageNet Normalization
train_dir = os.path.join("dataset", "train")
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)

print(f"Loaded {len(train_dataset)} images across {len(train_dataset.classes)} classes: {train_dataset.classes}")

Loaded 10906 images across 6 classes: ['freshapples', 'freshbanana', 'freshoranges', 'rottenapples', 'rottenbanana', 'rottenoranges']


In [3]:
# 3. Synchronized Labels.json Export (List-based for direct indexing)
os.makedirs("models", exist_ok=True)
with open("models/labels.json", "w") as f:
    json.dump(train_dataset.classes, f, indent=4)
print("Saved synchronized labels.json")

Saved synchronized labels.json


In [4]:
# 4. Model Architecture & Phase 1 Setup (Frozen Backbone)
model = models.resnet18(weights="DEFAULT")

# Freeze all pretrained backbone parameters
for param in model.parameters():
    param.requires_grad = False

# Replace final fully connected layer (automatically has requires_grad = True)
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

# Optimizer passes only the parameters of model.fc
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

In [5]:
# 5. Two-Stage Training Loop

PHASE1_EPOCHS = 5   # Warm up head
PHASE2_EPOCHS = 10  # Fine-tune full network
TOTAL_EPOCHS = PHASE1_EPOCHS + PHASE2_EPOCHS

print("=== Starting Phase 1: Training Classification Head Only ===")

for epoch in range(TOTAL_EPOCHS):
    # Transition to Phase 2: Unfreeze backbone
    if epoch == PHASE1_EPOCHS:
        print("\n=== Starting Phase 2: Unfreezing Backbone for Fine-Tuning ===")
        
        # Unfreeze all layers
        for param in model.parameters():
            param.requires_grad = True
            
        # Switch to differential learning rates with a smaller LR for backbone
        optimizer = optim.Adam([
            {'params': model.conv1.parameters(), 'lr': 1e-5},
            {'params': model.bn1.parameters(),   'lr': 1e-5},
            {'params': model.layer1.parameters(), 'lr': 1e-5},
            {'params': model.layer2.parameters(), 'lr': 1e-5},
            {'params': model.layer3.parameters(), 'lr': 5e-5},
            {'params': model.layer4.parameters(), 'lr': 5e-5},
            {'params': model.fc.parameters(),     'lr': 1e-4}
        ], weight_decay=1e-4)

    model.train()
    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        if (batch_idx + 1) % 10 == 0:
            phase_name = "Phase 1 (Frozen)" if epoch < PHASE1_EPOCHS else "Phase 2 (Fine-tune)"
            print(f"[{phase_name}] Epoch [{epoch+1}/{TOTAL_EPOCHS}] | Batch [{batch_idx+1}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"--> Epoch [{epoch+1}/{TOTAL_EPOCHS}] Completed | Average Loss: {epoch_loss:.4f}\n")

print("Training finished successfully!")

=== Starting Phase 1: Training Classification Head Only ===
[Phase 1 (Frozen)] Epoch [1/15] | Batch [10/341] | Loss: 1.5280
[Phase 1 (Frozen)] Epoch [1/15] | Batch [20/341] | Loss: 1.0246
[Phase 1 (Frozen)] Epoch [1/15] | Batch [30/341] | Loss: 0.9676
[Phase 1 (Frozen)] Epoch [1/15] | Batch [40/341] | Loss: 0.9241
[Phase 1 (Frozen)] Epoch [1/15] | Batch [50/341] | Loss: 0.5850
[Phase 1 (Frozen)] Epoch [1/15] | Batch [60/341] | Loss: 0.5834
[Phase 1 (Frozen)] Epoch [1/15] | Batch [70/341] | Loss: 0.4059
[Phase 1 (Frozen)] Epoch [1/15] | Batch [80/341] | Loss: 0.4401
[Phase 1 (Frozen)] Epoch [1/15] | Batch [90/341] | Loss: 0.3602
[Phase 1 (Frozen)] Epoch [1/15] | Batch [100/341] | Loss: 0.4198
[Phase 1 (Frozen)] Epoch [1/15] | Batch [110/341] | Loss: 0.4551
[Phase 1 (Frozen)] Epoch [1/15] | Batch [120/341] | Loss: 0.3175
[Phase 1 (Frozen)] Epoch [1/15] | Batch [130/341] | Loss: 0.3338
[Phase 1 (Frozen)] Epoch [1/15] | Batch [140/341] | Loss: 0.2878
[Phase 1 (Frozen)] Epoch [1/15] | Batch

In [6]:
# 6. Clean ONNX Export
onnx_path = "models/fruit_model.onnx"
model.eval()
dummy_input = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    dynamo=False
)

print(f"Model successfully exported to {onnx_path}")

C:\Users\ROMAX\AppData\Local\Temp\ipykernel_26924\1389346760.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Model successfully exported to models/fruit_model.onnx
